# Generating Ground Truth Data

To evaluate search, we need a dataset of queries where we know which
document is the correct answer. This is called ground truth (or gold
standard) data.

For each query in our ground truth dataset, we know which document in
the knowledge base is relevant. When we run a search, we check whether
the results include the correct document.

There are several ways to get ground truth data:

- Human annotators look at documents and write queries (best quality, expensive)
- Collect real user queries and label them (requires a running system)
- Generate synthetic data with an LLM (what we'll do)

We don't have a production system yet, so we'll use an LLM to generate
questions. For each FAQ document, we ask the LLM to create 5 questions
that this document would answer. Then we know that for each generated
question, the source document is the correct answer.

## Loading the documents

We'll use helper files from module 01 and this module.

If you don't have them in your notebook directory, download them:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget ${PREFIX}/cohorts/2026/01-agentic-rag/code/ingest.py
wget ${PREFIX}/cohorts/2026/01-agentic-rag/code/rag_helper.py
wget ${PREFIX}/cohorts/2026/04-evaluation/code/evaluation_utils.py
```

In [22]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/04-evaluation/code/evaluation_utils.py

--2026-09-21 23:01:37--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 3073 (3.0K) [text/plain]
Saving to: ‘evaluation_utils.py.1’

evaluation_utils.py 100%[===================>]   3.00K  --.-KB/s    in 0s      

2026-09-21 23:01:38 (42.3 MB/s) - ‘evaluation_utils.py.1’ saved [3073/3073]



Then load the FAQ data:

In [23]:
from ingest import load_faq_data
documents = load_faq_data()

We'll generate questions only for the LLM Zoomcamp FAQ. The full FAQ
dataset contains documents from multiple courses. Generating five
questions for every document would take longer and cost more.

In [24]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

153

We'll use these documents from now on so let's name them as `documents`

In [25]:
documents = documents_llm

Each document already has an `id` field:

In [26]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [46]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

The ID becomes the label in our ground truth dataset. We generate
questions from a document, so we know that this document holds the
answer. Later, search evaluation checks whether search brings back the
document with this ID.

This is why every record needs a stable ID. If you can't uniquely
identify a document, you can't tell whether search retrieved the right
one. When you build your own evaluation set, assign an ID to each record
in your knowledge base first.

## Generating questions with structured output

We use an LLM to generate questions for each document.

This is the first time we're using structured output in the course.
With structured output, we ask the LLM to return data in a specific
format instead of free-form text. For example, instead of getting a
paragraph that contains questions, we can ask for a Python object with
a `questions` field.

This is useful when code will process the output. The model returns the
same structure every time. We can access the generated questions
directly instead of parsing text manually.


We want the output as a list of strings, so we define that structure
with a Pydantic model:

In [27]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

The instructions for the LLM:

In [28]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

We ask the LLM to use different wording from the original document.
This makes the evaluation more realistic - real users won't phrase
their questions the same way as the FAQ.


Call the LLM for one document:

In [36]:
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.responses import ResponseInputItemParam

load_dotenv()
openai_client = OpenAI()

Prepare the document as JSON:

In [30]:
import json

user_prompt = json.dumps(doc)

In [31]:
print(user_prompt)

{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."}


Create the messages:

In [37]:
messages: list[ResponseInputItemParam] = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [38]:
messages

[{'role': 'developer',
  'content': "You emulate a student who's taking our course.\nFormulate 5 questions this student might ask based on a FAQ record. The record\nshould contain the answer to the questions, and the questions should be complete and not too short.\nIf possible, use as fewer words as possible from the record.\n\nThe output should resemble how people ask questions\non the internet. Not too formal, not too short, not too long."},
 {'role': 'user',
  'content': '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'}]

Until now we called `responses.create` and read `response.output_text`.
For structured output we switch to `responses.parse` and pass
`text_format=Questions`, which tells the API to return our class instead
of free text.

Call the model:

In [ ]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [35]:
response

ParsedResponse[TypeVar](id='resp_0166f456c1c7dff2006ab1b75561f487d29b02f63a9d491628', created_at=1790031701.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ParsedResponseOutputMessage[TypeVar](id='msg_0166f456c1c7dff2006ab1b7568b4087d295c54bfa0fbaf725', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"questions":["Can I still join the course if I found it late?","If I join after the course has started, can I still get a certificate?","What do I need to do to be eligible for a certificate if I joined late?","Is it okay to start the course now, or is it already too late?","Do project submissions have a deadline for getting the certificate?"]}', type='output_text', logprobs=[], parsed=Questions(questions=['Can I still join the course if I found it late?', 'If I join after the course has started, can I still get a certificate?', 'What do I need to do to be eligible for a certificate if I j

The parsed object is available in `response.output_parsed`:

In [44]:
result = response.output_parsed
result

Questions(questions=['Can I still join the course if I found it late?', 'If I join after the course has started, can I still get a certificate?', 'What do I need to do to be eligible for a certificate if I joined late?', 'Is it okay to start the course now, or is it already too late?', 'Do project submissions have a deadline for getting the certificate?'])

We can access the list directly:

In [43]:
if result is not None:
    print(result.questions)

['Can I still join the course if I found it late?', 'If I join after the course has started, can I still get a certificate?', 'What do I need to do to be eligible for a certificate if I joined late?', 'Is it okay to start the course now, or is it already too late?', 'Do project submissions have a deadline for getting the certificate?']


You should see 5 questions that relate to the first FAQ document.

## Reusable utilities

We'll need this pattern again in other evaluation sections today, so
we put it in a reusable helper.

It contains helper functions we'll reuse in this module:

- `llm_structured`: calls the OpenAI API with structured output
- `llm_structured_retry`: retries structured-output calls when a
  request fails
- `calc_price`: calculates the price from token usage
- `calc_total_price`: calculates the total price from multiple usage
  objects
- `map_progress`: runs work in parallel and tracks progress. We'll use it
  in the next lesson.

Import the structured-output helper:

In [47]:
from evaluation_utils import llm_structured

Use it on the same document:

In [48]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

if result is not None:
    print(result.questions)

['I just found this course — is it still possible to join now?', 'Can I start the course late, or am I too late to sign up?', 'If I join after the course has already started, will I still be able to get a certificate?', 'What do I need to do to qualify for the certificate if I’m joining now?', 'Is there a deadline for submitting the project if I want the certificate?']


## Tracking cost

The response also contains token usage:

In [49]:
assert usage is not None
usage.input_tokens, usage.output_tokens

(207, 97)

As in the agents module, we calculate the price from `response.usage`.

Import the price helper:

In [50]:
from evaluation_utils import calc_price

Calculate the cost of this call:

In [51]:
cost = calc_price(usage)
cost

{'input_cost': 0.00015525, 'output_cost': 0.0004365, 'total_cost': 0.00059175}

Now convert these questions into ground truth records:

In [ ]:
records = []

assert result is not None
for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it still possible to join now?',
  'document': '74eb249bbf'},
 {'question': 'Can I start the course late, or am I too late to sign up?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for the certificate if I’m joining now?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for submitting the project if I want the certificate?',
  'document': '74eb249bbf'}]

Each record has two fields:

- `question`: the question generated by the LLM
- `document`: the ID of the FAQ document that should answer the question

The `document` field connects the generated question to the document
that contains the answer. Later, when we evaluate search, we'll ask the
search engine the generated question. Then we'll check if it retrieves
the document with this ID.

We now know how to generate and store questions for one document. In
the next lesson, we'll run this for all LLM Zoomcamp FAQ documents and
save the full ground truth dataset.